# deepCommodity — train everything on Colab

**Runtime → Change runtime type → GPU** before running.

This notebook is 4 cells: clone, cd, mount Drive, train everything. The `dc-train-all` make target auto-detects Drive and saves all artifacts there.

In [ ]:
!git clone -b master --single-branch https://github.com/juan-garassino/deepCommodity.git

In [ ]:
%cd deepCommodity

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!make dc-train-all OUTPUT_DIR=/content/drive/MyDrive/deepCommodity_outputs

## Variants

If you want a smoke test first (~10 min on T4):
```
!make dc-train-all-fast
```

If you want to override symbols / dates / model size:
```
!make dc-train-all \
    OUTPUT_DIR=/content/drive/MyDrive/deepCommodity_outputs \
    SYMBOLS=BTC,ETH,SOL,TIA,INJ \
    EQUITY_SYMBOLS=AAPL,NVDA,SOFI,IONQ \
    DAYS=730 EQUITY_DAYS=720 \
    SEQ_LEN=336 HORIZON=48 \
    EPOCHS=50 BATCH_SIZE=256
```

## What gets saved

Under `/content/drive/MyDrive/deepCommodity_outputs/`:
- `bars/<SYMBOL>.csv` — crypto OHLCV (Binance public)
- `equity_bars/<SYMBOL>.csv` — US equity OHLCV (yfinance)
- `orderflow/<SYMBOL>.csv` — crypto trade tape, last 24h
- `models/<SYMBOL>.pt` — price transformer per symbol (crypto + equity)
- `models/<SYMBOL>.orderflow.pt` — order-flow transformer per crypto symbol

On your laptop, sync the Drive folder, then inference runs on CPU:
```
cp -r ~/Drive/deepCommodity_outputs/models data/
python tools/forecast.py --input crypto.json --model ensemble --news-input news.json
```